In [2]:
from scipy.stats import chi2
from heritage import *


def compare_univariate_vs_marked_amplitude(
    events,
    marks,
    end_times,
    univariate_kwargs=None,
    marked_kwargs=None,
    tol=1e-6,
):
    if univariate_kwargs is None:
        univariate_kwargs={}

    if marked_kwargs is None:
        marked_kwargs={}

    for forbidden in ("alpha_l2","beta_l2","eta_l2"):
        if univariate_kwargs.get(forbidden,0.0)!=0.0:
            raise ValueError("Le test de vraisemblance doit être fait sans pénalisation L2.")
        if marked_kwargs.get(forbidden,0.0)!=0.0:
            raise ValueError("Le test de vraisemblance doit être fait sans pénalisation L2.")

    univ=UnivariateHawkesMLE(**univariate_kwargs)
    univ.fit(events,end_times=end_times)

    x0_marked=np.array(
        [
            univ.mu_,
            univ.alpha_,
            univ.beta_,
            0.0,
        ],
        dtype=float,
    )

    marked=UnivariateMarkedAmplitudeHawkesMLE(**marked_kwargs)
    marked.fit(
        events,
        marks=marks,
        end_times=end_times,
        x0=x0_marked,
    )

    ll_univ=float(univ.log_likelihood_)
    ll_marked=float(marked.log_likelihood_)

    diff=ll_marked-ll_univ
    lrt_stat=2.0*max(diff,0.0)
    p_value=float(1.0-chi2.cdf(lrt_stat,df=1))

    return {
        "univariate_model":univ,
        "marked_model":marked,
        "ll_univariate":ll_univ,
        "ll_marked":ll_marked,
        "ll_difference":diff,
        "marked_likelihood_at_least_univariate":bool(diff>=-tol),
        "lrt_stat":float(lrt_stat),
        "df":1,
        "p_value":p_value,
        "eta_hat":float(marked.eta_),
        "optimization_warning":None if diff>=-tol else (
            "La vraisemblance marquée est plus faible que l'univariée. "
            "Cela indique probablement un problème d'optimisation, de pénalisation, "
            "de bornes sur eta, ou de données différentes."
        ),
    }

In [3]:
import numpy as np
from scipy.stats import chi2


def simulate_marked_amplitude_hawkes(
    T,
    mu=0.2,
    alpha=0.3,
    beta=1.5,
    eta=0.7,
    seed=123,
    max_events=200000,
):
    rng=np.random.default_rng(seed)

    if mu<=0 or alpha<0 or beta<=0:
        raise ValueError("Il faut mu>0, alpha>=0 et beta>0.")

    theoretical_branching=alpha*np.exp(0.5*eta**2)

    if theoretical_branching>=1:
        raise ValueError(
            "Paramètres instables pour marks N(0,1). "
            "Il faut alpha*exp(eta^2/2)<1."
        )

    n_immigrants=rng.poisson(mu*T)

    events=list(rng.uniform(0.0,T,size=n_immigrants))
    marks=list(rng.normal(0.0,1.0,size=n_immigrants))

    parent_idx=0

    while parent_idx<len(events):
        parent_time=events[parent_idx]
        parent_mark=marks[parent_idx]

        mean_children=alpha*np.exp(eta*parent_mark)
        n_children=rng.poisson(mean_children)

        if n_children>0:
            delays=rng.exponential(scale=1.0/beta,size=n_children)
            child_times=parent_time+delays
            mask=child_times<=T
            child_times=child_times[mask]

            if len(child_times)>0:
                child_marks=rng.normal(0.0,1.0,size=len(child_times))

                events.extend(child_times.tolist())
                marks.extend(child_marks.tolist())

        parent_idx+=1

        if len(events)>max_events:
            raise RuntimeError(
                "Trop d'événements simulés. "
                "Le processus est peut-être proche de l'instabilité."
            )

    events=np.asarray(events,dtype=float)
    marks=np.asarray(marks,dtype=float)

    order=np.argsort(events)

    return events[order],marks[order]

In [4]:
T=500.0

events,marks=simulate_marked_amplitude_hawkes(
    T=T,
    mu=0.15,
    alpha=0.35,
    beta=1.2,
    eta=0.8,
    seed=42,
)

print("Nombre d'événements :",len(events))
print("Premiers events :",events[:5])
print("Premiers marks :",marks[:5])

Nombre d'événements : 166
Premiers events : [ 3.68113488 15.40891728 16.33672044 21.90188289 29.15137084]
Premiers marks : [-1.37668615  1.06598023  0.75086899 -0.33903308 -0.05378255]


In [5]:
test=compare_univariate_vs_marked_amplitude(
    events,
    marks=marks,
    end_times=T,
    univariate_kwargs={
        "n_starts":5,
        "random_state":123,
    },
    marked_kwargs={
        "n_starts":10,
        "random_state":123,
        "eta_bounds":(-5.0,5.0),
    },
)

print(test["ll_univariate"])
print(test["ll_marked"])
print(test["ll_difference"])
print(test["marked_likelihood_at_least_univariate"])
print(test["lrt_stat"])
print(test["p_value"])
print(test["eta_hat"])
print(test["optimization_warning"])

-274.05464403724807
-266.84855127711387
7.206092760134197
True
14.412185520268395
0.00014684899074401958
0.8063928233107804
None
